## 0. Mount Google Drive & copy data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
from pathlib import Path

# Copy data files from repo (uploaded or cloned) into working dir
WORK_DIR = Path("/content/nemotron_challenge")
WORK_DIR.mkdir(exist_ok=True)
os.chdir(WORK_DIR)

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

# If data files are in Drive, copy them; otherwise assume they're already here
DRIVE_DATA = Path("/content/drive/MyDrive/nemotron_challenge/data")
if DRIVE_DATA.exists():
    for f in ["test.csv", "train.csv"]:
        src = DRIVE_DATA / f
        if src.exists() and not (DATA_DIR / f).exists():
            shutil.copy2(src, DATA_DIR / f)
            print(f"Copied {f} from Drive")

# Verify model weights path
MODEL_PATH = "/content/hf_models"
assert Path(MODEL_PATH).exists(), f"Model not found at {MODEL_PATH}!"
print(f"Model path OK: {MODEL_PATH}")
!ls -lh {MODEL_PATH} | head -20

In [ ]:
!pip install -q unsloth transformers peft accelerate bitsandbytes sentencepiece protobuf
!pip install -q flash-attn --no-build-isolation

# Nemotron Baseline Inference & Submission

Load the base NVIDIA-Nemotron-3-Nano-30B-A3B model (no fine-tuning),
run inference on `test.csv`, and package a baseline `submission.zip`
with a fresh (identity) LoRA adapter.

In [ ]:
import csv
import json
import re
import zipfile
import torch
from pathlib import Path
from unsloth import FastLanguageModel

# ── Config ──────────────────────────────────────────────────────────
MODEL_PATH     = "/content/hf_models"
TEST_CSV       = "data/test.csv"
TRAIN_CSV      = "data/train.csv"
MAX_SEQ_LEN    = 8192
MAX_NEW_TOKENS = 7680

# LoRA config (must match challenge constraints)
LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

ADAPTER_DIR    = Path("baseline_adapter")
SUBMISSION_ZIP = Path("submission.zip")

## 1. Load base model (4-bit quantized + Flash Attention)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_PATH,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,          # auto-detect (bf16 on A100)
    load_in_4bit   = True,
    attn_implementation = "flash_attention_2",
)
print(f"Model loaded: {MODEL_PATH}")
print(f"Vocab size  : {tokenizer.vocab_size}")
print(f"Device      : {model.device}")
print(f"Dtype       : {model.dtype}")

## 2. Create baseline LoRA adapter (untrained = identity)

A freshly initialised LoRA adapter has B=0, so `ΔW = A·B = 0`.
The model behaves exactly like the base model — this gives us the true baseline.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

total  = sum(p.numel() for p in model.parameters())
train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {train:,} / {total:,}  ({train/total*100:.2f}%)")

## 3. Inference helpers

In [ ]:
SYSTEM_PROMPT = """You are a concise mathematical reasoning assistant. Solve puzzles step-by-step.

## Instructions
1. Study the input \u2192 output examples to find the hidden rule.
2. Test your hypothesis on 2-3 examples.
3. Apply the rule to the target input.
4. Give your final answer as $\\boxed{<answer>}$.

Be direct. Skip filler. End with $\\boxed{<answer>}$.
"""

BOXED_RE = re.compile(r'\\boxed\{([^}]*)\}')


def extract_last_boxed(text: str) -> str | None:
    matches = BOXED_RE.findall(text)
    return matches[-1].strip() if matches else None


def run_inference(model, tokenizer, prompt: str) -> tuple[str, str | None]:
    """Generate a response and extract the boxed answer."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt + "\n\nSolve and end with $\\boxed{<answer>}$."}
    ]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    FastLanguageModel.for_inference(model)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            temperature    = 0.6,
            top_p          = 0.95,
            do_sample      = True,
            use_cache      = True,
        )

    generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    answer = extract_last_boxed(generated)
    return generated, answer

## 4. Load test data & ground truth

In [ ]:
# Load test puzzles
with open(TEST_CSV) as f:
    test_rows = list(csv.DictReader(f))
print(f"Test puzzles: {len(test_rows)}")

# Load ground truth from train.csv for validation
gt = {}
with open(TRAIN_CSV) as f:
    for r in csv.DictReader(f):
        gt[r["id"]] = r["answer"]

for r in test_rows:
    r["gt_answer"] = gt.get(r["id"], "??")
    print(f"  {r['id']}: GT = {r['gt_answer']}")

## 5. Run baseline inference on test puzzles

In [ ]:
results = []

for i, row in enumerate(test_rows):
    print(f"\n{'='*60}")
    print(f"Puzzle {i+1}/{len(test_rows)}  |  ID: {row['id']}  |  GT: {row['gt_answer']}")
    print(f"{'='*60}")

    generated, predicted = run_inference(model, tokenizer, row["prompt"])

    correct = (predicted == row["gt_answer"]) if predicted else False
    # Numeric tolerance check
    if not correct and predicted:
        try:
            p, a = float(predicted), float(row["gt_answer"])
            if a != 0:
                correct = abs(p - a) / abs(a) <= 0.01
            else:
                correct = abs(p - a) <= 1e-12
        except ValueError:
            pass

    status = "\u2705 CORRECT" if correct else "\u274c WRONG"
    print(f"\nPredicted : {predicted}")
    print(f"Expected  : {row['gt_answer']}")
    print(f"Status    : {status}")
    print(f"Tokens gen: ~{len(generated) // 4}")
    print(f"\n--- Generated (first 1000 chars) ---")
    print(generated[:1000])

    results.append({
        "id": row["id"],
        "predicted": predicted or "",
        "gt_answer": row["gt_answer"],
        "correct": correct,
        "generated": generated,
    })

# Summary
n_correct = sum(1 for r in results if r["correct"])
print(f"\n{'='*60}")
print(f"BASELINE ACCURACY: {n_correct}/{len(results)} ({n_correct/len(results)*100:.1f}%)")
print(f"{'='*60}")

## 6. Save baseline adapter & create submission.zip

In [ ]:
# Save the untrained adapter (identity / no-op)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))

print("Adapter files:")
for f in sorted(ADAPTER_DIR.iterdir()):
    mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<40} {mb:.1f} MB")

In [ ]:
ALLOWED_FILES = {"adapter_model.safetensors", "adapter_config.json"}

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ADAPTER_DIR.iterdir():
        if f.name in ALLOWED_FILES:
            zf.write(f, arcname=f.name)
            print(f"  Added: {f.name}")

size_mb = SUBMISSION_ZIP.stat().st_size / 1024 / 1024
print(f"\nsubmission.zip created ({size_mb:.1f} MB)")

# Verify contents
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    print("Contents:")
    for info in zf.infolist():
        print(f"  {info.filename:<40} {info.file_size/1024/1024:.1f} MB")

In [ ]:
# Save results for reference
with open("data/baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Results saved to data/baseline_results.json")

# Copy submission to Drive for easy download
DRIVE_OUT = Path("/content/drive/MyDrive/nemotron_challenge")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(SUBMISSION_ZIP, DRIVE_OUT / "submission.zip")
print(f"Copied submission.zip to {DRIVE_OUT}")

print("\n" + "="*60)
print("DONE \u2014 upload submission.zip to the challenge")
print("="*60)